# Comparable companies from website-text embeddings

First matching pass: treat one company as if it were private and rank the
other companies in the supplied list by how similar their **website text** is,
using sentence embeddings.

- **Inputs:** name + website only (Aziz's rule). The text is the cleaned page
  text from the website scrape (`website_scrape/scrape_websites.py`), the same text Codex
  reads. **No Codex fields are used here.**
- **Financial fields** (revenue, EBITDA, margin, debt/revenue, EV/EBITDA) are
  never loaded: `company_table()` reads only the non-financial columns.
- **SIC codes** are used only to check the results, never in the score.
- Each run saves its inputs, settings and results to a new timestamped folder,
  `outputs-local/scraps/comps_first_pass/notebook_embed/<run time>/`, so no run is overwritten.

Sections follow the write-up template: candidate peer pool → information
collected → matching method → results (+ checks).

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()
if not (HERE / "comps_similarity.py").exists():     # opened from the repo root
    HERE = HERE / "pre-v0" / "scraps" / "comps_first_pass"
sys.path.insert(0, str(HERE))
import comps_similarity as cs                       # also puts website_scrape/ and src/ on the path
from project_data import load_manifest, snapshot_path, sha256
from scrape_websites import output_paths
from step2_extract_profiles import build_prompt, mask, name_variants

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---- settings -------------------------------------------------------------
TARGET = "PCTY"     # ticker of the company treated as private
TOP_K = 25          # comps to report
EVIDENCE_K = 15     # comps to explain with matching sentences
# ---------------------------------------------------------------------------

RUN_AT = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = cs.OUT_ROOT / "notebook_embed" / RUN_AT
OUT.mkdir(parents=True, exist_ok=True)
print("embedding model:", cs.EMBED_MODEL)
print("outputs:", OUT)

embedding model: sentence-transformers/all-mpnet-base-v2
outputs: C:\Users\micha\Qual2Cast\outputs-local\task6\notebook_embed\20261018T190530Z


## 1. Candidate peer pool

The supplied universe: every row of the snapshot in `data_manifest.json`
(checked against its checksum), minus the target. There is no pre-filter by
SIC or size. A company only drops out when its website gave too little usable
text; each one is listed with its reason.

In [2]:
manifest = load_manifest()
snapshot = manifest["required_snapshot"]
snap_file = snapshot_path()                          # stops if the checksum doesn't match
companies = cs.company_table()
assert not {"target_revenue", "target_ebitda", "target_margin",
            "target_debt_to_revenue", "target_ev_to_ebitda"} & set(companies.columns)
print(f"snapshot {snapshot}: {len(companies)} companies ({snap_file.name})")

texts = cs.page_texts()                              # cleaned, name-masked page text
universe, excluded = cs.build_universe(texts, companies, "pages")
print(f"matchable: {len(universe)}   excluded: {len(excluded)}   "
      f"candidates for {TARGET}: {len(universe) - 1}")
assert TARGET in set(universe.target_ciq_ticker), f"{TARGET} is not matchable"

reason_group = excluded.excluded_because.str.replace(r"\s*\(\d+ words\)", "", regex=True)
display(reason_group.value_counts().rename("companies").to_frame())
display(excluded)

snapshot company_list_v3: 560 companies (company_list_v3_websites_fixed.csv)


matchable: 534   excluded: 26   candidates for PCTY: 533


,companies
excluded_because,
too little text after cleaning,13
status: investor_site,4
status: http_403,4
status: robots_disallowed,4
status: blocked,1


,IQ_COMPANY_ID,target_ciq_ticker,target_company_name,excluded_because
50,IQ100918,AVA,Avista Corporation,status: investor_site
57,IQ285544,BBWI,"Bath & Body Works, Inc.",status: http_403
65,IQ255902,BKH,Black Hills Corporation,status: investor_site
91,IQ7612672,CARS,Cars.com Inc.,status: http_403
118,IQ27023,CPRT,"Copart, Inc.",too little text after cleaning (33 words)
134,IQ116958995,CURA,"Curaleaf Holdings, Inc.",too little text after cleaning (20 words)
152,IQ299160,DORM,"Dorman Products, Inc.",status: robots_disallowed
163,IQ109103,EMN,Eastman Chemical Company,status: robots_disallowed
168,IQ27933,EHTH,"eHealth, Inc.",too little text after cleaning (29 words)
174,IQ382171572,A950140,"Englewood Lab, Inc.",too little text after cleaning (93 words)


## 2. Nonfinancial information collected

Per company, up to 3 pages of its own website: the homepage plus the
best-scoring products / solutions / about / industries / customers pages.
Pages were fetched as plain HTML, browser-rendered, stealth-browser or Wayback
Machine copies, with robots.txt honored. Raw HTML is kept in
`outputs-local/website_scrape/raw_html/`.

Before matching, Task 3 cleaning removed site chrome and any sentence that
reveals a public company or carries financial or size information (tickers,
investor and earnings talk, revenue and headcount figures, deal history,
founding dates). Every removal is logged with its reason in
`page_text/<IQ_ID>.json`. The company's own name and domain are masked as "the
Company". Here, sentences that appear on 3 or more companies' sites (shared
templates) and leftover boilerplate are also dropped.

Below: the source pages for every matchable company, and the target's text.

In [3]:
page_dir = output_paths(cs.SCRAPE_OUT).page_text

def load_record(iq_id):
    return json.loads((page_dir / f"{iq_id}.json").read_text(encoding="utf-8"))

def page_sentences(rec, keep):
    # (sentence, source URL) for the sentences the company was matched on
    urls = {p["page_id"]: p["url"] for p in rec["pages"]}
    names = name_variants(rec["company_name"], rec["website"])
    out = []
    for pid, text in build_prompt(rec)[1].items():
        for s in cs.to_sentences(mask(text, names, [])[0]):
            if s in keep:
                out.append((s, urls[pid]))
    return out

sources = []
for r in universe.itertuples():
    rec = load_record(r.IQ_COMPANY_ID)
    for p in rec["pages"]:
        sources.append({"IQ_COMPANY_ID": r.IQ_COMPANY_ID, "ticker": r.target_ciq_ticker,
                        "page_id": p["page_id"], "category": p["category"], "url": p["url"],
                        "method": p["method"], "wayback_snapshot": p["snapshot"],
                        "fetched_at": p["fetched_at"], "raw_html_file": p["raw_html_file"]})
sources = pd.DataFrame(sources)
urls_by_ticker = sources.groupby("ticker")["url"].apply(" | ".join)

print("fetch method per company:")
display(universe["method"].value_counts().to_frame("companies"))
print("cleaned words per company:")
display(universe["clean_words"].describe().round(0).to_frame().T)

t = universe.index[universe.target_ciq_ticker == TARGET][0]
print(f"\n{TARGET} ({universe.target_company_name[t]}): {universe.clean_words[t]} words, pages:")
display(sources[sources.ticker == TARGET][["page_id", "category", "url", "method", "fetched_at"]])
print(universe.clean_text[t][:1500], "...")

fetch method per company:


,companies
method,
html,468
rendered,47
stealth,13
wayback,6


cleaned words per company:


,count,mean,std,min,25%,50%,75%,max
clean_words,534.0,1009.0,660.0,122.0,529.0,838.0,1328.0,3453.0



PCTY (Paylocity Holding Corporation): 1158 words, pages:


,page_id,category,url,method,fetched_at
1018,P1,home,https://www.paylocity.com/,rendered,2026-10-17T20:03:30Z
1019,P2,markets,https://www.paylocity.com/who-we-serve/,rendered,2026-10-17T20:03:34Z
1020,P3,offerings,https://www.paylocity.com/products/,rendered,2026-10-17T20:03:39Z


Unify HR, Finance, & IT HR & Payroll Simplified unifies HR, Finance, and IT on one HCM platform, so you can stop piecing systems together and start moving forward together. The next evolution of work is here to automate manual processes, reduce risk, and help you move faster. ONE experience can change everything. Power every team with a single software Our flexible solutions drive innovation, enable smarter decisions, transform workforces, fuel growth, and inspire confident leadership. Simplify payroll and filing taxes with an intuitive, modern platform and expert support, knowing your payroll is accurate each time. Transform spend management – real-time visibility, AI-powered automation, seamless integration, smarter workflows. Centralize asset management, access control, and service support – empowering IT to enable the business, not just support it. A Payroll bundle is any new purchase of Payroll, standalone or bundled with other products. “It was everything we needed, all in one pl

## 3. Matching method: embeddings

Each company's cleaned text is cut into 200-word chunks. Each chunk is
embedded with `all-mpnet-base-v2` (open-source sentence-transformers model,
run locally). The chunk vectors are averaged and L2-normalised into one vector
per company. Similarity is the cosine between the target's vector and each
candidate's, and candidates are ranked by it.

No LLM is used in the ranking. Vectors are cached in `outputs-local/scraps/comps_first_pass/cache/`
under the model name and a hash of the texts, so changed text never reuses old
vectors. The first run takes about 10 minutes on a laptop CPU.

In [4]:
E = cs.embed_texts(universe["clean_text"].tolist(), cs.OUT_ROOT / "cache")
print("embedding matrix:", E.shape)

embedding matrix: (534, 768)


## 4. Results: ranked comparables

Score = cosine similarity to the target (1 = identical direction). `z` is the
score in standard deviations above the mean candidate. `same_sic4` is shown
for checking only.

In [5]:
sims = E @ E[t]
others = np.arange(len(universe)) != t
z = (sims - sims[others].mean()) / sims[others].std()
order = [i for i in np.argsort(-sims) if i != t]

def ranked(idx_list):
    return pd.DataFrame({
        "rank": range(1, len(idx_list) + 1),
        "ticker": universe.target_ciq_ticker[idx_list].values,
        "company": universe.target_company_name[idx_list].values,
        "score": sims[idx_list].round(3),
        "z": z[idx_list].round(2),
        "sic4": universe.target_sic4_code[idx_list].values,
        "sic_industry": universe.target_primary_sic_industry[idx_list].values,
        "same_sic4": universe.target_sic4_code[idx_list].values == universe.target_sic4_code[t],
        "words": universe.clean_words[idx_list].values,
    })

all_ranked = ranked(order)                           # every candidate, saved below
comps = all_ranked.head(TOP_K).copy()
comps["source_urls"] = comps.ticker.map(urls_by_ticker)
print(f"{TARGET} (SIC {universe.target_sic4_code[t]}, {universe.target_primary_sic_industry[t]})")
display(comps.drop(columns="source_urls"))

PCTY (SIC 7370, Computer Programming, Data Processing, And Other Computer Related Services)


,rank,ticker,company,score,z,sic4,sic_industry,same_sic4,words
0,1,PAYC,"Paycom Software, Inc.",0.856,2.83,7370,"Computer Programming, Data Processing, And Other Computer Related Services",True,1319
1,2,ASUR,"Asure Software, Inc.",0.851,2.80,7370,"Computer Programming, Data Processing, And Other Computer Related Services",True,1629
2,3,BBSI,"Barrett Business Services, Inc.",0.837,2.71,7363,Help Supply Services,False,3225
3,4,ADP,"Automatic Data Processing, Inc.",0.829,2.66,7374,Computer Processing and Data Preparation and Processing Services,False,1008
4,5,HURN,Huron Consulting Group Inc.,0.792,2.42,8742,Management Consulting Services,False,966
5,6,INTU,Intuit Inc.,0.779,2.33,7372,Prepackaged Software,False,903
6,7,UPLD,"Upland Software, Inc.",0.768,2.26,7370,"Computer Programming, Data Processing, And Other Computer Related Services",True,1138
7,8,FA,First Advantage Corporation,0.739,2.07,7389,"Business Services, not elsewhere classified",False,1502
8,9,EVCM,EverCommerce Inc.,0.736,2.05,7370,"Computer Programming, Data Processing, And Other Computer Related Services",True,570
9,10,KFY,Korn Ferry,0.727,1.99,7361,Employment Agencies,False,1124


### Evidence: the closest sentences

Why each match: for the top comps, the pairs of sentences (target, comp) whose
own embeddings are most similar, with the page each comp sentence came from.
These are the facts a reviewer should check against the source pages in the
human sanity check.

In [6]:
from sentence_transformers import SentenceTransformer
st = SentenceTransformer(cs.EMBED_MODEL)

def sentence_vectors(i):
    rec = load_record(universe.IQ_COMPANY_ID[i])
    pairs = page_sentences(rec, set(universe.clean_sents[i]))
    vecs = st.encode([s for s, _ in pairs], normalize_embeddings=True) if pairs else np.zeros((0, E.shape[1]))
    return pairs, vecs

t_pairs, t_vecs = sentence_vectors(t)
evidence = []
for rank, i in enumerate(order[:EVIDENCE_K], 1):
    c_pairs, c_vecs = sentence_vectors(i)
    if not c_pairs:
        continue
    S = t_vecs @ c_vecs.T
    used_t, used_c = set(), set()
    for flat in np.argsort(-S, axis=None):               # best pairs, each sentence used once
        a, b = np.unravel_index(flat, S.shape)
        if a in used_t or b in used_c:
            continue
        used_t.add(a); used_c.add(b)
        evidence.append({"rank": rank, "ticker": universe.target_ciq_ticker[i],
                         "sentence_similarity": round(float(S[a, b]), 3),
                         "target_sentence": t_pairs[a][0], "target_url": t_pairs[a][1],
                         "comp_sentence": c_pairs[b][0], "comp_url": c_pairs[b][1]})
        if len(used_t) == 3:
            break
evidence = pd.DataFrame(evidence)
display(evidence[["rank", "ticker", "sentence_similarity", "target_sentence", "comp_sentence", "comp_url"]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,rank,ticker,sentence_similarity,target_sentence,comp_sentence,comp_url
0,1,PAYC,0.775,"brings teams together across HR, Finance, and IT.","supports organizations across company sizes, industries and roles — including HR, finance, IT decision-makers, execu...",https://www.paycom.com/about/
1,1,PAYC,0.762,"unifies HR, Finance, and IT on one HCM platform, so you can stop piecing systems together and start moving forward t...","is an HCM platform, with HR, time and labor, talent management and compliance built around payroll in the same datab...",https://www.paycom.com/about/
2,1,PAYC,0.753,automates every step of the employee lifecycle and keeps your team engaged along the way.,automates the employee life cycle — from hiring to payroll to time and labor — while reducing the manual work create...,https://www.paycom.com/about/
3,2,ASUR,0.793,HR & Payroll Simplified,Payroll processing and HR management,https://www.asuresoftware.com/
4,2,ASUR,0.770,"Employees expect the right pay, every time, and payroll admins are there to get it right every time.",Employees expect to be paid correctly.,https://www.asuresoftware.com/
5,2,ASUR,0.745,"brings HR, Finance, and IT together to empower teams, simplify processes, and deliver real results.","’s strategy and growth, focusing on building a platform that helps businesses manage payroll, HR, and compliance wit...",https://www.asuresoftware.com/about-us/
6,3,BBSI,0.779,"brings HR, Finance, and IT together to empower teams, simplify processes, and deliver real results.","helps fast-growing companies manage HR, payroll, benefits, and risk.",https://www.bbsi.com/about
7,3,BBSI,0.750,"From recruiting and onboarding to payroll and benefits administration,",Payroll Administration & Human Resources Consulting,https://www.bbsi.com/
8,3,BBSI,0.719,"unifies processes across HR, Finance, and IT so you can automate more work, stay in sync, and build a stronger organ...","helps organizations operate with confidence, free from administrative complexity.",https://www.bbsi.com/about
9,4,ADP,0.829,is the best solution for your business.,Let's find the perfect solution for your business.,https://www.adp.com/


## 5. Checks

- **Peer labels** (`peer_labels.csv`): graded peers written *before* any
  ranking was seen (2 = direct competitor, 1 = adjacent). NDCG@10/20 and
  recall@10/25, against the mean of 200 random orderings. Only PCTY has labels
  so far; add labels for a new target before running it.
- **SIC4, all companies**: how often each company's 5 nearest neighbours share
  its SIC4 code, against chance. A weak check, since SIC is coarse and sometimes
  wrong (e.g. CPI Card Group).
- **Text length**: if similarity mostly tracks how much text a company has,
  long sites would dominate every list.

In [7]:
labels = cs.load_peer_labels()
checks = {}
if TARGET in labels:
    peers = labels[TARGET]
    rng = np.random.default_rng(cs.SEED)
    cand = list(all_ranked.ticker)
    rand = pd.DataFrame([cs.metrics(list(rng.permutation(cand)), peers)
                         for _ in range(cs.RANDOM_ORDERINGS)]).mean()
    peer_val = pd.DataFrame({"embeddings": cs.metrics(cand, peers), "random": rand}).T.round(3)
    display(peer_val)
    checks["peer_labels"] = peer_val.to_dict("index")
    lab = pd.DataFrame({"peer": list(peers), "grade": list(peers.values())})
    lab["rank"] = lab.peer.map(dict(zip(all_ranked.ticker, all_ranked["rank"])))
    lab["note"] = np.where(lab["rank"].isna(), "not matchable", "")
    print("where each labeled peer landed:")
    display(lab.sort_values("rank"))
else:
    print(f"No peer labels for {TARGET}: write them in peer_labels.csv before looking at the list above.")

sic = universe["target_sic4_code"].values
SS = E @ E.T
np.fill_diagonal(SS, -np.inf)
top5 = np.argsort(-SS, axis=1)[:, :5]
rate = (sic[top5] == sic[:, None]).mean()
chance = np.mean([(sic == s).sum() - 1 for s in sic]) / (len(sic) - 1)
checks["sic4_top5"] = {"same_sic4": round(rate, 3), "chance": round(chance, 3), "lift": round(rate / chance, 1)}
checks["spearman_score_vs_words"] = round(pd.Series(sims[others]).corr(
    pd.Series(universe.clean_words.values[others]), method="spearman"), 3)
print(checks["sic4_top5"], " spearman(score, words):", checks["spearman_score_vs_words"])

,NDCG@10,NDCG@20,Recall@10,Recall@25
embeddings,0.868,0.897,0.636,0.909
random,0.019,0.024,0.022,0.046


where each labeled peer landed:


,peer,grade,rank,note
0,PAYC,2,1,
2,ASUR,2,2,
3,BBSI,1,3,
1,ADP,2,4,
5,INTU,1,6,
6,FA,1,8,
8,KFY,1,10,
7,HSTM,1,14,
9,HQI,1,16,
4,ALIT,1,21,


{'same_sic4': np.float64(0.239), 'chance': np.float64(0.017), 'lift': np.float64(13.9)}  spearman(score, words): 0.313


## 6. Save

Everything needed to reproduce or audit this run goes to the run folder:
the ranked list (all candidates and top K), evidence sentences, the exact text
each company was matched on, exclusions, source pages, and settings.

In [8]:
all_ranked.to_csv(OUT / f"ranking_{TARGET}.csv", index=False)
comps.to_csv(OUT / f"comps_{TARGET}.csv", index=False)
evidence.to_csv(OUT / f"evidence_{TARGET}.csv", index=False)
universe[["IQ_COMPANY_ID", "target_ciq_ticker", "status", "method", "clean_words", "clean_text"]] \
    .to_csv(OUT / "universe_text.csv", index=False)
excluded.to_csv(OUT / "excluded_companies.csv", index=False)
sources.to_csv(OUT / "source_pages.csv", index=False)
run_info = {
    "run_at": RUN_AT, "target": TARGET, "snapshot": snapshot,
    "snapshot_sha256": manifest["snapshots"][snapshot]["sha256"],
    "text_source": "Task 3 step 1 page text (cleaned, name-masked); no Codex fields",
    "embed_model": cs.EMBED_MODEL, "chunk_words": cs.CHUNK_WORDS,
    "shared_sentence_sites": cs.SHARED_SENTENCE_SITES, "min_words": cs.MIN_WORDS["pages"],
    "matchable": len(universe), "excluded": len(excluded), "checks": checks,
}
(OUT / "run_info.json").write_text(json.dumps(run_info, indent=1, default=str), encoding="utf-8")
print("saved to", OUT)
print(sorted(p.name for p in OUT.iterdir()))

saved to C:\Users\micha\Qual2Cast\outputs-local\task6\notebook_embed\20261018T190530Z
['comps_PCTY.csv', 'evidence_PCTY.csv', 'excluded_companies.csv', 'ranking_PCTY.csv', 'run_info.json', 'source_pages.csv', 'universe_text.csv']
